In [1]:
import os
from pydantic_ai import Agent
from pydantic_ai.capabilities import MCP
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

from IPython.display import display, Markdown

## Let us go one step higher in the abstraction by using Agents and letting them decide which tools to call rather than specifying them manually

In [2]:
agent = Agent(
    name="hpc-docs-agent",
    model=OpenAIResponsesModel(
        model_name="@vertexai/gemini-3.7-flash",
        provider=OpenAIProvider(
            base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
            api_key=os.getenv("PORTKEY_API_KEY"),
        ),
    ),
    instructions="Be concise and answer questions from retreived knowledge with tools.",
    capabilities=[
        MCP(
            url="https://mcp-gateway.apps.cloud.rt.nyu.edu/rts-docs-algolia-public/mcp",
            id="rts-docs-algolia-mcp",
            headers={
                "x-portkey-api-key": os.getenv("PORTKEY_API_KEY"),
            },
        ),
    ],
)

In [3]:
result = await agent.run("Login to HPC cluster from off campus") # await is added because the agent is run asynchronously

In [4]:
display(Markdown(result.output))

To log into the NYU HPC cluster (Torch) from off-campus:

### 1. Connect to the NYU VPN
You must first connect to the NYU network using the **NYU VPN**. 
* Set up and start your VPN client according to NYU IT instructions before attempting to connect.

---

### 2. Connect via SSH (Terminal)

#### **Mac / Linux / Windows (PowerShell or WSL):**
Open your terminal and run:
```bash
ssh <NetID>@login.torch.hpc.nyu.edu
```
*(Replace `<NetID>` with your NYU NetID)*

#### **Recommended SSH Config (`~/.ssh/config`):**
To avoid host-key warnings and improve stability, add the following to your `~/.ssh/config` file:
```text
Host dtn.torch.hpc.nyu.edu
    User <NetID>
    StrictHostKeyChecking no
    ServerAliveInterval 60
    ForwardAgent yes
    UserKnownHostsFile /dev/null
    LogLevel ERROR

Host torch login.torch.hpc.nyu.edu
    Hostname login.torch.hpc.nyu.edu
    User <NetID>
    StrictHostKeyChecking no
    ServerAliveInterval 60
    ForwardAgent yes
    UserKnownHostsFile /dev/null
    LogLevel ERROR

Host cs* cl* gr* ga* gh* gl*
    User <NetID>
    ProxyCommand ssh -W %h:%p torch-login
    ForwardAgent yes
```
*(Note: SSH keys are not supported on Torch due to security restrictions).*

---

### 3. Complete Two-Factor Authentication (MFA)
When prompted in the terminal:
1. Open [https://microsoft.com/devicelogin](https://microsoft.com/devicelogin).
2. Enter the **PIN** provided in your terminal prompt.
3. Sign in with your `<NetID>@nyu.edu` credentials and approve the **Duo MFA** prompt.
4. Return to your terminal and press **Enter** to complete the login.

---

### Alternative: Web Access via Open OnDemand (OOD)
While connected to the NYU VPN, you can also access the cluster via a web browser:
* Go to **[https://ood.torch.hpc.nyu.edu](https://ood.torch.hpc.nyu.edu)**.
* Under the **Clusters** menu, select **Torch Shell Access** for command-line access or launch interactive GUI applications (e.g., Jupyter, RStudio).

## Let's see how we can evaluate the performance of the Agent on this task by varying the LLM used. 
## Here we check if the output from the Agent contains the specific string `login.torch.hpc.nyu.edu` to ensure that the model did not hallucinate a new login address:

In [5]:
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Contains

# Create a dataset with test cases
dataset = Dataset(
    name='login-to-hpc',
    cases=[
        Case(
            name="use-gemini-3.7-flash",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-3.7-flash"
            },
        ),
        Case(
            name="use-gemini-2.5-flash-lite",
            inputs={
                "query": "Login to HPC cluster from off campus",
                "model": "@vertexai/gemini-2.5-flash-lite"
            },
        ),
    ],
    evaluators=[
        Contains(value='login.torch.hpc.nyu.edu', case_sensitive=True),
    ],
)

async def agent_task(inputs: dict) -> str:
    with agent.override(model=
                        OpenAIResponsesModel(
                            model_name=inputs["model"],
                            provider=OpenAIProvider(
                                base_url="https://ai-gateway.apps.cloud.rt.nyu.edu/v1/",
                                api_key=os.getenv("PORTKEY_API_KEY"),
                                ),
                        )
                       ):
        result = await agent.run(user_prompt=inputs["query"])
        return result.output


# Run the evaluation
report = await dataset.evaluate(agent_task)

# Print the results
report.print()

Output()

           Evaluation Summary: agent_task            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Case ID                   ┃ Assertions ┃ Duration ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━┩
│ use-gemini-3.7-flash      │ ✔          │    12.3s │
├───────────────────────────┼────────────┼──────────┤
│ use-gemini-2.5-flash-lite │ ✗          │     4.3s │
├───────────────────────────┼────────────┼──────────┤
│ Averages                  │ 50.0% ✔    │     8.3s │
└───────────────────────────┴────────────┴──────────┘

## Why did the case with `gemini-2.5-flash-lite` fail? Let's check the output from that run:

In [6]:
display(Markdown(report.cases[1].output))

I apologize for the errors in my previous response. I need to ensure I'm only returning text or calling a tool when appropriate.

To log in to the HPC cluster from off-campus, you will likely need to connect to the VPN first. Once connected to the VPN, you should be able to use SSH to log in to the cluster.

Here are the general steps:

1.  **Connect to the VPN:** Instructions for connecting to the NYU VPN can typically be found on the IT services website.
2.  **Open a terminal or SSH client:** On your local machine, open your preferred terminal application (like Terminal on macOS/Linux or PuTTY/Windows Terminal on Windows).
3.  **SSH into the cluster:** Use the SSH command with your NetID and the cluster's login node address. It will look something like this:

    ```bash
    ssh your_netid@cluster_login_node_address
    ```
    You may be prompted for your NetID password or Duo multi-factor authentication.

For specific details and instructions tailored to NYU's HPC environment, please refer to the official IT documentation. The previous search results indicated a relevant page on "HPC bursting to cloud" which might contain more detailed information on accessing resources remotely.

## The older, less capable model misunderstood the prompt and answered the question for a different HPC cluster (Cloud bursting). Explore how you may prevent this?